In [ ]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
heart_disease = fetch_ucirepo(id=45) 
  
# data (as pandas dataframes) 
X = heart_disease.data.features 
y = heart_disease.data.targets 
  
# metadata 
print(heart_disease.metadata) 
  
# variable information 
print(heart_disease.variables)

## Dicionário de Variáveis — Heart Disease Dataset (UCI, id=45)

Dataset da Cleveland Clinic Foundation, usado para prever a presença de doença cardíaca. Contém 303 instâncias, 13 variáveis preditoras (features) e 1 variável alvo (target).

| Variável | Tipo | Descrição |
|---|---|---|
| **age** | Feature (inteiro) | Idade do paciente, em anos |
| **sex** | Feature (categórica) | Sexo do paciente: `1` = masculino, `0` = feminino |
| **cp** | Feature (categórica) | Tipo de dor no peito (chest pain): `1` = angina típica, `2` = angina atípica, `3` = dor não anginosa, `4` = assintomático |
| **trestbps** | Feature (inteiro) | Pressão arterial em repouso (mm Hg), medida na admissão hospitalar |
| **chol** | Feature (inteiro) | Colesterol sérico, em mg/dl |
| **fbs** | Feature (categórica) | Glicemia em jejum > 120 mg/dl: `1` = verdadeiro, `0` = falso |
| **restecg** | Feature (categórica) | Resultado do eletrocardiograma em repouso: `0` = normal, `1` = anormalidade da onda ST-T, `2` = hipertrofia ventricular esquerda provável/definitiva (critério de Estes) |
| **thalach** | Feature (inteiro) | Frequência cardíaca máxima atingida durante o teste de esforço |
| **exang** | Feature (categórica) | Angina induzida por exercício: `1` = sim, `0` = não |
| **oldpeak** | Feature (contínua) | Depressão do segmento ST induzida pelo exercício em relação ao repouso |
| **slope** | Feature (categórica) | Inclinação do segmento ST no pico do exercício: `1` = ascendente, `2` = plano, `3` = descendente |
| **ca** | Feature (inteiro) | Número de vasos principais (0 a 3) coloridos por fluoroscopia |
| **thal** | Feature (categórica) | Resultado do exame de tálio: `3` = normal, `6` = defeito fixo, `7` = defeito reversível |
| **num** | Target (inteiro) | Diagnóstico de doença cardíaca (status angiográfico): `0` = ausência de doença, `1`–`4` = presença, com grau crescente de severidade |

**Observação:** para classificação binária (presença/ausência de doença), é comum transformar `num` em `0` (sem doença) vs. `1` (com doença), agrupando os valores `1`–`4`.


In [ ]:
import polars as pl

df = pl.concat(
    [pl.from_pandas(X), pl.from_pandas(y)],
    how="horizontal"
)
df.head()


In [ ]:
df.shape

In [ ]:
df = (
    df
    .drop_nulls()                                                      # remove null
    .filter(~pl.any_horizontal(pl.col(pl.Float32, pl.Float64).is_nan()))  # remove NaN
    .unique(maintain_order=True)                                       # remove duplicados
)
df.shape

In [ ]:
df.columns

In [ ]:
import pandas as pd
import category_encoders as ce

# ---------------------------------------------------------------------
# WOEEncoder — Weight of Evidence
#
# Substitui cada categoria por:   WOE = ln( P(cat | y=1) / P(cat | y=0) )
# ou seja, o log da razão de chances daquela categoria. Nasceu em credit scoring.
#
# Por que é interessante (vs. one-hot):
#   - gera 1 coluna por variável: não expande a dimensionalidade
#   - já sai na escala de log-odds, que casa direto com regressão logística
#   - o sinal é interpretável: WOE > 0 puxa para "tem doença", < 0 para "não tem"
#   - não assume ordem entre categorias (ao contrário do PolynomialEncoder)
#
# NOTA: o WOE é calculado a partir do alvo. Aqui ajustamos no dataset inteiro
# porque o objetivo é apenas demonstrar a codificação. Na etapa de modelagem,
# o fit do encoder precisa ficar dentro do Pipeline/validação cruzada,
# enxergando somente o fold de treino.
# ---------------------------------------------------------------------

pdf = df.to_pandas()          # category_encoders trabalha com pandas

cat_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']

X_in  = pdf.drop(columns='num')
y_bin = (pdf['num'] > 0).astype(int)      # WOE exige alvo BINÁRIO: 0 = sem doença, 1..4 -> 1

woe   = ce.WOEEncoder(cols=cat_cols, random_state=42)
X_woe = woe.fit_transform(X_in, y_bin)

print(f"colunas: {X_in.shape[1]} -> {X_woe.shape[1]} (WOE nao expande a dimensionalidade)")
print(f"shape: {X_woe.shape}\n")

# --- valores de WOE aprendidos: quanto cada categoria pesa a favor da doenca ---
for col in cat_cols:
    tab = (pd.DataFrame({'categoria': X_in[col], 'WOE': X_woe[col]})
             .drop_duplicates()
             .sort_values('WOE', ascending=False)
             .reset_index(drop=True))
    print(f"--- {col} ---")
    print(tab.to_string(index=False), "\n")

X_woe.head()

In [ ]:
# ---------------------------------------------------------------------
# DataFrame final: 13 features (categóricas já em WOE) + alvo binário
# ---------------------------------------------------------------------
df_encoded = X_woe.copy()
df_encoded['num'] = y_bin          # alinha pelo índice: 0 = sem doença, 1 = com doença

print(f"shape : {df_encoded.shape}")
print(f"nulos : {df_encoded.isna().sum().sum()}")
print(f"dtypes:\n{df_encoded.dtypes}\n")

# versão em polars, para seguir o restante do fluxo
df_encoded_pl = pl.from_pandas(df_encoded)

df_encoded.head()

In [ ]:
df_encoded.to_csv("heart_disease_woe.csv", sep=";", index=False, decimal=".")
